# Episode 18: Testing Agents

How do you know your agent actually works? Not "seems to work when I try it" — actually works, reliably, at 2 AM when you're not watching.

In this episode, you'll learn how to test agent behaviors, mock LLM calls, and evaluate quality across many scenarios.

## Why Testing Agents Is Hard

Testing agents is harder than testing functions. But there are practical approaches that work.

Traditional unit tests verify deterministic behavior: given input X, expect output Y. Agents break this model in several ways:

| Challenge | Why It's Hard |
|-----------|---------------|
| **Non-deterministic outputs** | Same prompt can produce different responses every time |
| **Multi-turn conversations** | Behavior depends on the full conversation history |
| **Emergent behavior** | Multi-agent interactions create behaviors you didn't explicitly program |
| **External dependencies** | LLM API calls are slow, expensive, and rate-limited |

Amazon learned this firsthand while scaling thousands of agentic systems since 2025 — evaluating agents requires fundamentally different strategies than evaluating traditional software. Instead of testing outputs, you test **behaviors**: Does the agent call the right tool? Does the handoff route to the correct agent? Does the conversation terminate within limits? Does the agent refuse out-of-scope requests?

In [ ]:
!pip install "ag2[openai]" python-dotenv pytest -q

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from autogen import AssistantAgent, ConversableAgent, LLMConfig

## Part 1: Mocking the LLM

The first practical step is replacing the real LLM with **deterministic responses** for testing. This gives you speed (no API calls, tests run in milliseconds), determinism (same mock, same response), zero cost, and reliability (no flaky tests from rate limits).

The following three pytest-style test functions demonstrate the pattern. In a real project, you'd save these in a `test_agents.py` file and run with `pytest`.

In [ ]:
# --- Setup: Define the agents we want to test ---


def create_customer_service_agent():
    """Create a customer service agent with a tool."""
    llm_config = LLMConfig({"model": "gpt-4o-mini"})

    def lookup_account(customer_id: str) -> str:
        """Look up a customer account."""
        return f"Account {customer_id}: Active, Balance: $150.00"

    agent = AssistantAgent(
        name="service_agent",
        system_message=(
            "You are a customer service agent. Use the lookup_account tool "
            "to find customer information when asked about accounts."
        ),
        llm_config=llm_config,
    )

    agent.register_for_llm()(lookup_account)
    return agent, lookup_account

In [ ]:
# Test 1: Verify the agent calls the right tool for a relevant query


def test_agent_calls_tool():
    """Test that the agent invokes the lookup_account tool for account queries."""
    agent, lookup_fn = create_customer_service_agent()

    user = ConversableAgent(name="user", human_input_mode="NEVER")

    result = user.run(
        agent,
        message="Can you look up account C-1234?",
        max_turns=3,
    )
    result.process()

    # Check that the tool was called by looking at chat history
    messages = result.messages
    tool_calls_found = any(
        "lookup_account" in str(msg) or "C-1234" in str(msg) for msg in messages
    )
    assert tool_calls_found, "Agent should have called lookup_account tool"
    print("PASSED: test_agent_calls_tool")


# Run the test (in a real project, pytest runs this automatically)
test_agent_calls_tool()

In [ ]:
# Test 2: Verify handoff routes correctly


def test_handoff_routes_correctly():
    """Test that a billing query gets routed to the billing agent."""
    from autogen.agentchat import run_group_chat
    from autogen.agentchat.group import (
        AgentTarget,
        OnCondition,
        StringLLMCondition,
        RevertToUserTarget,
    )
    from autogen.agentchat.group.patterns import DefaultPattern

    llm_config = LLMConfig({"model": "gpt-4o-mini"})

    triage = AssistantAgent(
        name="triage_agent",
        system_message=(
            "You are a triage agent. Route billing questions to billing_agent. "
            "Route technical questions to tech_agent."
        ),
        llm_config=llm_config,
    )
    billing = AssistantAgent(
        name="billing_agent",
        system_message="You handle billing questions. Be concise.",
        llm_config=llm_config,
    )
    tech = AssistantAgent(
        name="tech_agent",
        system_message="You handle technical questions. Be concise.",
        llm_config=llm_config,
    )

    user = AssistantAgent(name="user", human_input_mode="NEVER")

    triage.handoffs.add_llm_conditions(
        [
            OnCondition(
                target=AgentTarget(billing),
                condition=StringLLMCondition(
                    prompt="When the query is about billing or charges."
                ),
            ),
            OnCondition(
                target=AgentTarget(tech),
                condition=StringLLMCondition(
                    prompt="When the query is about technical issues."
                ),
            ),
        ]
    )
    billing.handoffs.set_after_work(RevertToUserTarget())
    tech.handoffs.set_after_work(RevertToUserTarget())

    pattern = DefaultPattern(
        initial_agent=triage,
        agents=[triage, billing, tech],
        user_agent=user,
        group_manager_args={"llm_config": llm_config},
    )

    result = run_group_chat(
        pattern=pattern,
        messages="I was charged twice on my credit card.",
        max_rounds=4,
    )
    result.process()

    assert (
        result.last_speaker == "billing_agent" or "billing" in str(result).lower()
    ), f"Billing query should route to billing_agent. Last agent: {result.last_speaker}"
    print("PASSED: test_handoff_routes_correctly")


test_handoff_routes_correctly()

In [ ]:
# Test 3: Verify conversation terminates within limits


def test_agent_terminates():
    """Test that a conversation ends within the specified max_turns."""
    llm_config = LLMConfig({"model": "gpt-4o-mini"})

    agent = AssistantAgent(
        name="bounded_agent",
        system_message="You are a helpful assistant. Be concise.",
        llm_config=llm_config,
    )

    max_turns = 4
    user = AssistantAgent(name="user", human_input_mode="NEVER")

    result = user.run(
        agent,
        message="Tell me a very long story about dragons.",
        max_turns=max_turns,
    )
    result.process()

    message_count = len(result.messages)
    assert (
        message_count <= max_turns * 2
    ), f"Expected at most {max_turns * 2} messages, got {message_count}"
    print(
        f"PASSED: test_agent_terminates ({message_count} messages in {max_turns} max turns)"
    )


test_agent_terminates()

## Part 2: Evaluation

Testing individual behaviors is necessary but not sufficient. You also need to measure **overall quality** across many test cases.

The pattern is straightforward: define a set of test cases with expected outcomes, run them all, and measure the pass rate. Think of it as a scoring rubric for your agent system.

In [ ]:
from autogen.agentchat import run_group_chat
from autogen.agentchat.group import (
    AgentTarget,
    OnCondition,
    StringLLMCondition,
    RevertToUserTarget,
)
from autogen.agentchat.group.patterns import DefaultPattern

# Define test cases: input query and which agent should handle it
test_cases = [
    {"input": "What's my billing status?", "expected_agent": "billing_agent"},
    {"input": "My laptop won't turn on", "expected_agent": "tech_agent"},
    {"input": "What are your hours?", "expected_agent": "general_agent"},
    {"input": "I need a refund", "expected_agent": "billing_agent"},
    {"input": "How do I reset my password?", "expected_agent": "tech_agent"},
]


def create_triage_system():
    """Create a triage agent with specialist handoffs."""
    llm_config = LLMConfig({"model": "gpt-4o-mini"})

    triage = AssistantAgent(
        name="triage_agent",
        system_message=(
            "Route billing/refund questions to billing_agent. "
            "Route technical/password/device questions to tech_agent. "
            "Route general questions to general_agent."
        ),
        llm_config=llm_config,
    )
    billing = AssistantAgent(
        name="billing_agent",
        system_message="Handle billing questions. Be concise.",
        llm_config=llm_config,
    )
    tech = AssistantAgent(
        name="tech_agent",
        system_message="Handle technical questions. Be concise.",
        llm_config=llm_config,
    )
    general = AssistantAgent(
        name="general_agent",
        system_message="Handle general questions. Be concise.",
        llm_config=llm_config,
    )

    user = AssistantAgent(name="user", human_input_mode="NEVER")

    triage.handoffs.add_llm_conditions(
        [
            OnCondition(
                target=AgentTarget(billing),
                condition=StringLLMCondition(
                    prompt="When the query is about billing, charges, or refunds."
                ),
            ),
            OnCondition(
                target=AgentTarget(tech),
                condition=StringLLMCondition(
                    prompt="When the query is about technical issues, devices, or passwords."
                ),
            ),
            OnCondition(
                target=AgentTarget(general),
                condition=StringLLMCondition(
                    prompt="When the query is a general inquiry."
                ),
            ),
        ]
    )
    billing.handoffs.set_after_work(RevertToUserTarget())
    tech.handoffs.set_after_work(RevertToUserTarget())
    general.handoffs.set_after_work(RevertToUserTarget())

    pattern = DefaultPattern(
        initial_agent=triage,
        agents=[triage, billing, tech, general],
        user_agent=user,
        group_manager_args={"llm_config": llm_config},
    )
    return pattern

In [ ]:
# Run the evaluation loop
passed = 0
failed = 0
results = []

for i, test in enumerate(test_cases):
    pattern = create_triage_system()

    result = run_group_chat(
        pattern=pattern,
        messages=test["input"],
        max_rounds=4,
    )
    result.process()

    expected = test["expected_agent"]
    routed_correctly = result.last_speaker == expected

    if routed_correctly:
        passed += 1
        status = "PASS"
    else:
        failed += 1
        status = "FAIL"

    results.append(
        {
            "input": test["input"],
            "expected": expected,
            "got": result.last_speaker,
            "status": status,
        }
    )
    print(
        f"  [{status}] '{test['input']}' -> expected: {expected}, got: {result.last_speaker}"
    )

print(
    f"\nResults: {passed}/{len(test_cases)} passed ({100 * passed / len(test_cases):.0f}%)"
)

## Part 3: Preview — `ag2 test` and `ag2 arena`

The AG2 CLI is adding built-in testing tools that automate much of what we just did manually.

### `ag2 test`

`ag2 test` automates assertion-based evaluation. You define test cases in a YAML file and run them from the command line:

```yaml
# tests/routing_tests.yaml
tests:
  - name: billing_routing
    input: "I was double-charged"
    assert:
      - agent_involved: billing_agent
      - max_turns: 3

  - name: tech_routing
    input: "My app keeps crashing"
    assert:
      - agent_involved: tech_agent
```

```bash
ag2 test --config tests/routing_tests.yaml
```

### `ag2 arena`

For comparing agent configurations head-to-head, `ag2 arena` runs both agents on the same inputs and uses an LLM judge to determine which performs better, producing ELO-style rankings.

```bash
ag2 arena --agent-a config_v1.yaml --agent-b config_v2.yaml --eval-set questions.jsonl
```

> These features are coming in a future AG2 release. Stay tuned!

## Testing Best Practices

### Regression Testing

When you fix a bug, add a test case that reproduces it. Over time, you build a **golden dataset** of known-good behaviors that catches regressions before they ship.

### Golden Datasets

Maintain a curated set of input/output pairs that represent expected agent behavior. Run these on every code change.

### CI Integration

```yaml
# .github/workflows/test-agents.yml
- name: Run agent tests
  run: pytest tests/test_agents.py -v
  env:
    OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
```

Use mocked LLM responses for CI — they're fast, free, and deterministic. Run real LLM tests on a schedule (nightly, for example) to catch model behavior changes.

## Try It Yourself

Write 3 test cases for your Episode 10 customer service system:

1. A billing question routes to the billing specialist
2. A technical question routes to the tech specialist
3. The system terminates within a reasonable number of turns

In [ ]:
# Your code here — write test cases for your customer service system

## What's Next

You can now observe, secure, and test your agents. In **Episode 19**, you'll learn to **control your agent costs** — track token usage, use model tiering, and optimize spending without sacrificing quality.